# M1 (TCN baseline) tuning comparison

Two Optuna tuning runs of `tcn_HPT_binary.py` that share the **same trial sequence** during the random-startup phase (Trials 0-9, identical Optuna seed) but differ in the training class-balance ratio.

| | Run A | Run B |
|---|---|---|
| Train segments | 134,957 | 220,653 |
| Train ictal % | 48.5% | 29.7% |
| Ictal:non-ictal ratio | 1:1.06 | 1:2.37 |
| Validation partition | 4,298,154 (0.27% ictal) | identical |

Validation is identical, so this is a clean A/B test of the **train downsampling ratio**.

Source logs:
- Run A: `tcn_HPT_binary.py` log started 2026-04-26 03:23:28 (A100-PCIE-40GB, T_120 corpus)
- Run B: `tcn_HPT_binary.py` log started 2026-04-17 23:44:19 (V100-PCIE-32GB, current pipeline corpus)

## Notes on trial coverage

- **Trial 6** was pruned by Optuna's MedianPruner in both runs (no completion line in either log). Plotted as NaN to show the gap.
- **Trials 10+** are *not* directly comparable: the TPE sampler exits the random-startup phase at trial 10 and starts adapting based on prior F1 results, which diverge between runs. Different HPs suggested per run from T10 onwards. Plot is therefore restricted to T0-T9.

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

## Data

Best F1 per Optuna trial during the random-startup phase (T0-T9), extracted from each tuning `.log`. Trial-level hyperparameters are identical between runs for these trials, so trial index maps to the same `(num_layers, kernel, num_filters, dropout, lr, wd, bs)` combination in both runs. Trial 6 was pruned in both -- recorded as NaN to visualise the gap.

In [ ]:
TRIALS = list(range(10))                              # T0..T9 (random-startup phase, comparable HPs)

# Trial 6 was pruned by MedianPruner in both runs -- represented as NaN
RUN_A_BEST_F1 = [0.5231, 0.5631, 0.5722, 0.5318, 0.5222,
                 0.5365, np.nan, 0.5456, 0.5607, 0.5368]

RUN_B_BEST_F1 = [0.5612, 0.7006, 0.6667, 0.6049, 0.5610,
                 0.5803, np.nan, 0.6811, 0.6494, 0.6708]

RUN_A_LABEL = "1:1.06 ictal:non-ictal (134,957 segments)"
RUN_B_LABEL = "1:2.37 ictal:non-ictal (220,653 segments)"

RUN_A_COLOR = "#d95f02"                               # orange (matches M3 plot)
RUN_B_COLOR = "#1b6ca8"                               # blue   (matches M3 plot)

OUTPUT_PATH = Path("m1_tuning_comparison.png")

## Helper: cumulative running maximum (NaN-aware)

For the running-best curve we want to ignore the pruned trial (T6 = NaN) rather than break the cumulative max. `np.fmax.accumulate` treats NaN as 'no observation'.

In [ ]:
def running_max_nan(values):
    """Element-wise running maximum that ignores NaN entries."""
    return np.fmax.accumulate(np.asarray(values, dtype=float))

## Plot: per-trial best F1 + running best (cumulative max)

In [ ]:
run_a = np.asarray(RUN_A_BEST_F1)
run_b = np.asarray(RUN_B_BEST_F1)
run_a_running = running_max_nan(run_a)
run_b_running = running_max_nan(run_b)

fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(13, 5.5), sharey=True)

# -- Panel A: per-trial best F1 ------------------------------------------
line_a, = ax_left.plot(TRIALS, run_a, marker="o", linewidth=1.8, markersize=8,
                       color=RUN_A_COLOR, label=RUN_A_LABEL)
line_b, = ax_left.plot(TRIALS, run_b, marker="s", linewidth=1.8, markersize=8,
                       color=RUN_B_COLOR, label=RUN_B_LABEL)
ax_left.set_title("Per-trial best validation F1", fontsize=12, fontweight="bold")
ax_left.set_xlabel("Optuna trial index")
ax_left.set_ylabel("Best validation macro F1 in trial")
ax_left.set_xticks(TRIALS)
ax_left.set_ylim(0.50, 0.80)
ax_left.grid(True, linestyle="--", alpha=0.4)
# Annotate the pruned trial
ax_left.axvline(x=6, color="grey", linestyle=":", alpha=0.5)
ax_left.text(6, 0.51, "T6 pruned\n(both runs)", fontsize=8, color="grey",
             ha="center", va="bottom")

# -- Panel B: cumulative running best (Optuna's tuning curve) ------------
ax_right.plot(TRIALS, run_a_running, marker="o", linewidth=2.2, markersize=8,
              color=RUN_A_COLOR, label=RUN_A_LABEL)
ax_right.plot(TRIALS, run_b_running, marker="s", linewidth=2.2, markersize=8,
              color=RUN_B_COLOR, label=RUN_B_LABEL)
ax_right.set_title("Running best (cumulative max across trials)",
                   fontsize=12, fontweight="bold")
ax_right.set_xlabel("Optuna trial index")
ax_right.set_xticks(TRIALS)
ax_right.grid(True, linestyle="--", alpha=0.4)

# -- Single shared legend at the top of the figure -----------------------
fig.legend(handles=[line_a, line_b], labels=[RUN_A_LABEL, RUN_B_LABEL],
           loc="upper center", bbox_to_anchor=(0.5, 0.94), ncol=2,
           fontsize=10, framealpha=0.95)

fig.suptitle(
    "M1 (TCN baseline) tuning -- effect of training class-balance ratio\n"
    "Identical Optuna seed and trial sequence (T0-T9 random-startup phase); "
    "identical full-validation partition (4,298,154 segments, 0.27% ictal)",
    fontsize=12, y=1.04)

# Reserve top whitespace for the shared legend so it doesn't overlap the panels
plt.tight_layout(rect=[0, 0, 1, 0.90])
fig.savefig(OUTPUT_PATH, dpi=150, bbox_inches="tight")
print(f"Saved {OUTPUT_PATH.resolve()}")
plt.show()

In [ ]:
run_b_running

## Summary statistics

Computed over the 9 completed trials per run (T6 pruned in both).

In [ ]:
# Mask out NaN (pruned trial) for fair stats
mask = ~np.isnan(run_a) & ~np.isnan(run_b)
a = run_a[mask]
b = run_b[mask]

print(f"Run A (1:1.06): mean F1 = {a.mean():.4f} | "
      f"max = {a.max():.4f} (T{int(np.where(run_a == a.max())[0][0])})")
print(f"Run B (1:2.37): mean F1 = {b.mean():.4f} | "
      f"max = {b.max():.4f} (T{int(np.where(run_b == b.max())[0][0])})")
print(f"Mean gap (B - A): {(b - a).mean():+.4f}")
print(f"Run B beats Run A on {int((b > a).sum())}/{len(a)} comparable trials")